In [1]:
import numpy

In [2]:
import P04_Parser

In [3]:
%matplotlib qt

In [4]:
import matplotlib.pyplot as plt

In [5]:
# plt.hist(data.data[:, 100, 100], bins=100);


In [6]:
data = P04_Parser.HvScan.from_file(r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2B_49eV_LV_temp_series\FeSi_2B_hv_49_LV_GX_20K_0001Swept.ibw")

In [7]:
from igor2 import binarywave
import numpy as np

def load_ibw_swept_as_kmap(file):
    """Swept (2D) .ibw cut -> Kmap with 3 identical theta_y slices, so P04_Parser plotting works."""
    wave = binarywave.load(file)["wave"]
    data = np.array(wave["wData"], dtype=float)     # (energy, angle)
    sfa = wave["wave_header"]["sfA"]
    sfb = wave["wave_header"]["sfB"]
    energy = sfb[0] + sfa[0] * np.arange(data.shape[0])
    theta = sfb[1] + sfa[1] * np.arange(data.shape[1])
    data_map = np.repeat(data[:, :, np.newaxis], 3, axis=2)
    return P04_Parser.Kmap(data_map, energy, theta, np.array([-1, 0, 1]))

In [8]:
import P04_Parser

data = load_ibw_swept_as_kmap(
    r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2B_49eV_LV_temp_series\FeSi_2B_hv_49_LV_GX_20K_0001Swept.ibw"
)



data.plot("all")

c:\Users\lorentsen\anaconda3\envs\arpes\Lib\site-packages\hyperspy\signal.py:4404: RuntimeWarning: Mean of empty slice
  function(
c:\Users\lorentsen\anaconda3\envs\arpes\Lib\site-packages\hyperspy\signal.py:4382: RuntimeWarning: Mean of empty slice
  function(


(SpanROI(left=44.05, right=44.15),
 SpanROI(left=-16.3384, right=-9.7082),
 SpanROI(left=-1, right=-0.6))

In [10]:
kmap = P04_Parser.Kmap.from_file(r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2A_day1\FeSi_2A_kmap_hv_49_LH_GM_0001.zip")


In [11]:
kmap

SES Kmap
Region Name: kmap
Photon Energy: 48.9992978109 eV
Lens Mode: DA30L_01
Pass Energy: 10
User comments: ['T=140.6K cryostat, 149.5K coldfinger, sample estimated at 151.0K', 'Polarization Vertical (undulator gap/phase 27.1 / 42.0)', 'Exit slit h x v = 50 x 99', 'Spot size=10.0 um x 5.0 um, beamline dE = 3.2meV', 'Pressure = 9.69e-11mBar', 'Ring current 499.9mA']

In [12]:
import hyperspy.api as hs


# 1. Pick the region: a circle on the kmap and an energy span on the EDC
roi  = hs.roi.CircleROI()   # drag it to an area with a clear Fermi edge (or set cx, cy, r)
span = hs.roi.SpanROI()     # drag it so it covers the edge, with some flat region on each side
edc, span = kmap.visualize_fermi_fit_region(angle_roi=roi, energy_span=span)

# 2. Fit, once the ROI and span are where you want them


In [13]:
Ef = kmap.fit_fermi_energy(edc, span, plot=True, print_verbose_output=True)

Fermi Energy: 44.176 eV
K_bT:  128.27 meV
Fermi width temperature:  422.229 K
90-10 Fermi width: 159.90 meV
Energy Resolution: 0.10 meV


In [14]:
# optional: fix_temp=25 fixes kT to the sample temperature in K, so only the resolution is fitted

# 3. Convert to k-space first, then shift the energy axis
kmap.convert_to_k_space()
kmap.convert_to_binding_energy(Ef)


In [17]:
from P04_Parser.Overlay import SquareBZ

spans, sigs = kmap.plot(type="all", return_sig=True)
sig_kmap = sigs[0]                   # the constant-energy map

bz = SquareBZ(a_0=4.5, rotation=0, repeat_x=[-1, 0], repeat_y=[-1, 0],
              origin=[0, 0], color="tab:blue")
bz.add_to_plot(sig_kmap, mark_high_symmetry=True,
               labels=["Γ", "M", "X", "R"],   # rename the points if you like, e.g. ["Γ","M","X","R"]
               label_offset=[0.05, 0])
